# HeatLens pooled — Atlanta + Chicago

Ek model, dono train cities. Chicago photos segment, Atlanta fractions pehle se.

1. Runtime → Change runtime type → **T4 GPU** → Save
2. Drive `MyDrive/heatlens/` pe rakho:
   - `labels_atlanta.csv` (Mac se `data/labels.csv` — Atlanta fractions already filled)
   - `delivery_chicago_full.zip`
3. **Runtime → Run all**

Nikalega: `labels.csv`, `coefficients.json`, `segments.json` (download + Drive `heatlens/`).
Mac pe copy: `data/labels.csv`  `data/coefficients.json`  `data/segments.json`

In [ ]:
!pip -q install -U "transformers>=4.44,<5"
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Runtime → T4 GPU, then Runtime → Run all again."

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil, zipfile

DRIVE = Path("/content/drive/MyDrive/heatlens")
ATL_CSV = DRIVE / "labels_atlanta.csv"
if not ATL_CSV.is_file():
    ATL_CSV = DRIVE / "labels.csv"
CHI_ZIP = DRIVE / "delivery_chicago_full.zip"
WORK = Path("/content/heatlens_data")

assert ATL_CSV.is_file(), f"Upload Atlanta labels to {DRIVE}/labels_atlanta.csv (copy of data/labels.csv)"
assert CHI_ZIP.is_file(), f"Upload zip to {CHI_ZIP}"

shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
with zipfile.ZipFile(CHI_ZIP) as zf:
    zf.extractall(WORK)

csvs = list(WORK.rglob("labels_chicago*.csv"))
photo_dirs = [p for p in WORK.rglob("photos") if p.is_dir()]
assert csvs and photo_dirs
CHI_LABELS, PHOTOS = csvs[0], photo_dirs[0]
print("atlanta", ATL_CSV)
print("chicago labels", CHI_LABELS)
print("chicago photos", PHOTOS, "n=", len(list(PHOTOS.glob("*.jpg"))))

In [ ]:
from datetime import datetime, timezone

CANOPY_IDS = frozenset({4, 9, 17})
ASPHALT_IDS = frozenset({6})
SKY_IDS = frozenset({2})
BUILDING_IDS = frozenset({1})
LEAF_ON = frozenset({6, 7, 8})
SUMMER_W = 6.0
FRAC_COLS = ("canopy_frac", "asphalt_frac", "sky_frac", "building_frac")

def fractions_from_class_ids(class_ids):
    counts = {"canopy": 0, "asphalt": 0, "sky": 0, "building": 0, "n": 0}
    for raw in class_ids:
        cid = int(raw)
        counts["n"] += 1
        if cid in CANOPY_IDS:
            counts["canopy"] += 1
        elif cid in ASPHALT_IDS:
            counts["asphalt"] += 1
        elif cid in SKY_IDS:
            counts["sky"] += 1
        elif cid in BUILDING_IDS:
            counts["building"] += 1
    n = counts["n"]
    t = float(n)
    return {
        "canopy_frac": round(counts["canopy"] / t, 4),
        "asphalt_frac": round(counts["asphalt"] / t, 4),
        "sky_frac": round(counts["sky"] / t, 4),
        "building_frac": round(counts["building"] / t, 4),
    }

def _solve(matrix, rhs):
    n = len(rhs)
    a = [row[:] + [rhs[i]] for i, row in enumerate(matrix)]
    for col in range(n):
        pivot = max(range(col, n), key=lambda r: abs(a[r][col]))
        a[col], a[pivot] = a[pivot], a[col]
        scale = a[col][col]
        for j in range(col, n + 1):
            a[col][j] /= scale
        for row in range(n):
            if row == col:
                continue
            factor = a[row][col]
            for j in range(col, n + 1):
                a[row][j] -= factor * a[col][j]
    return [a[i][n] for i in range(n)]

def wols(x_rows, y, weights):
    p = len(x_rows[0])
    xtx = [[0.0] * p for _ in range(p)]
    xty = [0.0] * p
    for x, yi, w in zip(x_rows, y, weights):
        for i in range(p):
            xty[i] += w * x[i] * yi
            for j in range(p):
                xtx[i][j] += w * x[i] * x[j]
    for i in range(p):
        xtx[i][i] += 1e-6
    return _solve(xtx, xty)

def predict_dt(c, row):
    return (
        c["intercept"]
        + c["canopy"] * float(row["canopy_frac"])
        + c["asphalt"] * float(row["asphalt_frac"])
        + c["sky"] * float(row["sky_frac"])
        + c["building"] * float(row["building_frac"])
    )

def mae(yt, yp):
    return sum(abs(a - b) for a, b in zip(yt, yp)) / len(yt)

def r2(yt, yp):
    mean = sum(yt) / len(yt)
    ss_tot = sum((y - mean) ** 2 for y in yt)
    ss_res = sum((y - yhat) ** 2 for y, yhat in zip(yt, yp))
    return 1.0 if ss_tot == 0 and ss_res == 0 else (0.0 if ss_tot == 0 else 1.0 - ss_res / ss_tot)

def has_fracs(row):
    try:
        for c in FRAC_COLS:
            raw = row.get(c)
            if raw is None or raw == "":
                return False
            if not (0.0 <= float(raw) <= 1.0):
                return False
        return True
    except (TypeError, ValueError):
        return False

def capture_month(raw):
    try:
        x = float(raw)
    except (TypeError, ValueError):
        return None
    sec = x / 1000.0 if x > 1e10 else x
    return datetime.fromtimestamp(sec, tz=timezone.utc).month

def is_leaf_on(row):
    return capture_month(row.get("captured_at")) in LEAF_ON

print("helpers ok")

In [ ]:
import csv

atl = list(csv.DictReader(ATL_CSV.open()))
chi = list(csv.DictReader(CHI_LABELS.open()))
assert all(r.get("city", "").lower() == "atlanta" for r in atl), "Atlanta CSV city column must be atlanta"
assert all(r.get("city", "").lower() == "chicago" for r in chi), "Chicago CSV city column must be chicago"
assert all(has_fracs(r) for r in atl), "Atlanta labels must already have fractions (use data/labels.csv)"

photos = {p.stem for p in PHOTOS.glob("*.jpg")}
miss = [r["image_id"] for r in chi if r["image_id"] not in photos]
dts = [float(r["delta_t"]) for r in chi if str(r.get("delta_t") or "").strip() != ""]
print(f"atlanta={len(atl)} chicago={len(chi)} photos={len(photos)} missing_jpg={len(miss)} chi_delta_t={len(dts)}")
assert not miss and len(dts) == len(chi)

In [ ]:
from PIL import Image
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from tqdm.auto import tqdm

MODEL_ID = "nvidia/segformer-b0-finetuned-ade-512-512"
processor = SegformerImageProcessor.from_pretrained(MODEL_ID)
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_ID).to("cuda")
model.eval()

todo = [i for i, r in enumerate(chi) if not has_fracs(r)]
print("chicago to segment", len(todo), "/", len(chi))
for i in tqdm(todo):
    row = chi[i]
    image = Image.open(PHOTOS / f"{row['image_id']}.jpg").convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = logits.argmax(dim=1)[0].detach().cpu().reshape(-1).tolist()
    row.update(fractions_from_class_ids(pred))

assert all(has_fracs(r) for r in chi)
print("chicago filled", len(chi))

In [ ]:
import json
from collections import Counter

by_id = {r["image_id"]: r for r in atl}
by_id.update({r["image_id"]: r for r in chi})
rows = list(by_id.values())
usable = [r for r in rows if has_fracs(r)]
print("pooled", len(usable), dict(Counter(r["city"] for r in usable)))

OUT = Path("/content/labels.csv")
fields = []
for r in usable:
    for k in r:
        if k not in fields:
            fields.append(k)
for c in FRAC_COLS:
    if c not in fields:
        fields.append(c)
with OUT.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    w.writeheader()
    w.writerows(usable)
print("wrote", OUT)

def x_full(row):
    return [1.0, float(row["canopy_frac"]), float(row["asphalt_frac"]), float(row["sky_frac"]), float(row["building_frac"])]

def x_nc(row):
    return [1.0, float(row["asphalt_frac"]), float(row["sky_frac"]), float(row["building_frac"])]

def slope(xs, ys, weights):
    wsum = sum(weights)
    mx = sum(w * x for w, x in zip(weights, xs)) / wsum
    my = sum(w * y for w, y in zip(weights, ys)) / wsum
    num = sum(w * (x - mx) * (y - my) for w, x, y in zip(weights, xs, ys))
    den = sum(w * (x - mx) ** 2 for w, x in zip(weights, xs))
    return 0.0 if den <= 1e-12 else num / den

train_rows = [r for r in usable if r.get("split") == "train"]
test_rows = [r for r in usable if r.get("split") == "test"]
leaf = [r for r in train_rows if is_leaf_on(r)]
print("train", len(train_rows), "test", len(test_rows), "leaf_on", len(leaf))

w = [SUMMER_W if is_leaf_on(r) else 1.0 for r in train_rows]
y = [float(r["delta_t"]) for r in train_rows]
if len(leaf) >= 20:
    bnc = wols([x_nc(r) for r in train_rows], y, w)
    resid, cans, lw = [], [], []
    for r in leaf:
        pred = bnc[0] + bnc[1] * float(r["asphalt_frac"]) + bnc[2] * float(r["sky_frac"]) + bnc[3] * float(r["building_frac"])
        resid.append(float(r["delta_t"]) - pred)
        cans.append(float(r["canopy_frac"]))
        lw.append(SUMMER_W)
    canopy = slope(cans, resid, lw)
    intercept, asphalt, sky, building = bnc[0], bnc[1], bnc[2], bnc[3]
else:
    b = wols([x_full(r) for r in train_rows], y, w)
    intercept, canopy, asphalt, sky, building = b[0], b[1], b[2], b[3], b[4]
if canopy > 0:
    print("canopy", round(canopy, 4), "clipped to 0")
    bnc = wols([x_nc(r) for r in train_rows], y, w)
    intercept, asphalt, sky, building = bnc[0], bnc[1], bnc[2], bnc[3]
    canopy = 0.0

coef = {
    "intercept": intercept,
    "canopy": canopy,
    "asphalt": asphalt,
    "sky": sky,
    "building": building,
    "target_canopy_frac": 0.40,
}

def split_metrics(part):
    if not part:
        return None
    yt = [float(r["delta_t"]) for r in part]
    yp = [predict_dt(coef, r) for r in part]
    return {"n": len(part), "mae_c": round(mae(yt, yp), 4), "r_squared": round(r2(yt, yp), 4)}

metrics = {
    "train": split_metrics(train_rows),
    "test": split_metrics(test_rows),
    "fitted_at": datetime.now(timezone.utc).isoformat(),
    "leaf_on_train": len(leaf),
    "by_city": dict(Counter(r["city"] for r in usable)),
    "note": "B1 pooled Atlanta+Chicago; summer-weighted; canopy clipped <= 0.",
}
payload = dict(coef)
payload["metrics"] = metrics
Path("/content/coefficients.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")

segments = []
for r in usable:
    segments.append({
        "image_id": r["image_id"],
        "lat": float(r["lat"]),
        "lon": float(r["lon"]),
        "city": r["city"],
        "block_id": r.get("block_id") or "",
        "delta_t": float(r["delta_t"]),
        "canopy_frac": float(r["canopy_frac"]),
        "asphalt_frac": float(r["asphalt_frac"]),
        "sky_frac": float(r["sky_frac"]),
        "building_frac": float(r["building_frac"]),
        "split": r.get("split") or "train",
        "source": r.get("source") or "mapillary+fortyguard",
        "validated": True,
        "captured_at": r.get("captured_at") or "",
    })
Path("/content/segments.json").write_text(
    json.dumps({"version": 1, "generated_at": metrics["fitted_at"], "segments": segments}, indent=2),
    encoding="utf-8",
)
print(json.dumps(payload, indent=2))

In [ ]:
from google.colab import files

for name in ("labels.csv", "coefficients.json", "segments.json"):
    dest = Path("/content/drive/MyDrive/heatlens") / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(Path("/content", name).read_bytes())
    print("Drive", dest)
    files.download(f"/content/{name}")

print("Mac pe rakho: data/labels.csv  data/coefficients.json  data/segments.json")